# Fitting MOSK GPs using Stan

This notebook demonstrates using Stan in python for fitting MOSK GPs.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import nutpie
import arviz as az

os.environ["TBB_CXX_TYPE"] = "clang"  # or 'gcc'

In [2]:
filename_stub = "D2-Q1-seed25973-n200-mnar0.3-sigma0.1"

D = 2
Q = 1

input_df = pd.read_csv(f'data/{filename_stub}-sim.csv')
input_df.head()

,t,f,y,y_se,d
0,0.050505,-0.066424,-0.053149,0.1,1
1,0.060606,-0.238200,-0.253329,0.1,1
2,0.070707,-0.382027,-0.262070,0.1,1
3,0.101010,-0.420636,-0.355838,0.1,1
4,0.111111,-0.278230,-0.331435,0.1,1


In [3]:
input_params = pd.read_table(f'data/{filename_stub}-param.tsv')
input_params

,seed,w,Sigma,mu,theta,phi
0,25973,0.71,1.43,6.38,-2.60,0
1,25973,0.84,0.14,4.52,1.04,0


In [4]:
ns = input_df['d'].value_counts().to_numpy()
ns

array([70, 61])

In [5]:
compiled_model = nutpie.compile_stan_model(filename='stan/moskgp-zerophi.stan')

In [ ]:
compiled_model_with_data = compiled_model.with_data(
    D=D, 
    N=input_df.shape[0],
    ns=ns,
    d=input_df['d'],
    x=input_df['t'],
    y=input_df['y'],
    y_se=input_df['y_se']
)

In [ ]:
trace = nutpie.sample(
    compiled_model=compiled_model_with_data, 
    draws=1000,
    tune=1000,
    chains=6,
    cores=6, 
    seed=0, 
    progress_rate=1000
)

Progress,Draws,Divergences,Step Size,Gradients/Draw
,439,0,0.11,47
,484,0,0.19,55
,316,0,0.08,127
,90,0,0.02,490
,129,0,0.04,63
,71,0,0.03,127


In [ ]:
assert trace.sample_stats.diverging.sum() == 0
assert az.ess(trace).min().min() > 500
assert az.rhat(trace).max().max() > 1.02